# Model-Tests

In [2]:
%matplotlib inline
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("nelgiriyewithana/world-stock-prices-daily-updating")

print("Path to dataset files:", path)
data_path = path+"\World-Stock-Prices-Dataset.csv"
data = pd.read_csv(data_path)

<>:9: SyntaxWarning: invalid escape sequence '\W'
<>:9: SyntaxWarning: invalid escape sequence '\W'
C:\Users\s3phi\AppData\Local\Temp\ipykernel_5792\942347233.py:9: SyntaxWarning: invalid escape sequence '\W'
  data_path = path+"\World-Stock-Prices-Dataset.csv"
c:\Users\s3phi\anaconda3\envs\StockPredictor_venv_GPU\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\s3phi\.cache\kagglehub\datasets\nelgiriyewithana\world-stock-prices-daily-updating\versions\386


In [3]:
data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Brand_Name,Ticker,Industry_Tag,Country,Capital Gains
0,2025-06-20 00:00:00-04:00,6.170000,6.305000,6.080000,6.180000,12833288.0,0.0,0.0,peloton,PTON,fitness,usa,NaN
1,2025-06-20 00:00:00-04:00,249.289993,250.800003,246.619995,248.860001,3576600.0,0.0,0.0,hilton,HLT,hospitality,usa,NaN
2,2025-06-20 00:00:00-04:00,129.000000,132.809998,127.550003,128.240005,79819700.0,0.0,0.0,amd,AMD,technology,usa,NaN
3,2025-06-20 00:00:00-04:00,1234.449951,1248.500000,1224.349976,1231.410034,5341700.0,0.0,0.0,netflix,NFLX,entertainment,usa,NaN
4,2025-06-20 00:00:00-04:00,22.549999,22.549999,22.250000,22.290001,1227500.0,0.0,0.0,philips,PHG,technology,netherlands,NaN


In [4]:
#arr = data["Brand_Name"].unique()
#for i in arr:
#    print(i)

In [5]:
df = pd.DataFrame(data)
df_apple = df.loc[df["Brand_Name"] == "google", ["Date", "Close", "Brand_Name"]]
display(df_apple)

,Date,Close,Brand_Name
53,2025-06-20 00:00:00-04:00,166.639999,google
81,2025-06-20 00:00:00-04:00,166.639999,google
168,2025-06-18 00:00:00-04:00,173.320007,google
192,2025-06-18 00:00:00-04:00,173.320007,google
259,2025-06-17 00:00:00-04:00,175.949997,google
...,...,...,...
269213,2004-08-25 00:00:00-04:00,2.652653,google
269226,2004-08-24 00:00:00-04:00,2.624374,google
269293,2004-08-23 00:00:00-04:00,2.737738,google
269299,2004-08-20 00:00:00-04:00,2.710460,google


In [6]:
df_apple_preproc = df_apple.rename(columns={"Date": "timestamp", "Close": "target", "Brand_Name": "item_id"})
df_apple_preproc["item_id"] = df_apple_preproc['item_id'].astype("string")
df_apple_preproc["timestamp"] = df_apple_preproc['timestamp'].astype("string")
timecut = df_apple_preproc["timestamp"].str.slice(stop=10) #Cut hh:mm:ss and timezone
df_apple_preproc["timestamp"] = timecut
df_apple_preproc["timestamp"] = pd.to_datetime(timecut) #convert string into datetime64
df_apple_reordered =  df_apple_preproc[['item_id', 'timestamp', 'target']] #Reordering columns
df_irregular = TimeSeriesDataFrame(
    pd.DataFrame(df_apple_reordered)
)
df_regular = df_irregular.convert_frequency(freq="D")
df_filled = df_regular.fill_missing_values()
data = TimeSeriesDataFrame.from_data_frame(
    df = df_filled,
    id_column="item_id",
    timestamp_column="timestamp"
)

In [7]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

prediction_length = 10
train_data, test_data = data.train_test_split(prediction_length)

predictor = TimeSeriesPredictor(prediction_length=prediction_length, freq="D").fit(
    train_data, presets="bolt_base", hyperparameters={"Chronos": {"fine_tune": True, "fine_tune_lr": 1e-5, "fine_tune_steps": 7000}},
    time_limit=30,
)

Beginning AutoGluon training... Time limit = 30s
AutoGluon will save models to 'c:\workspace\CrimeMap\AutogluonModels\ag-20250623_074529'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.11
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.19045
CPU Count:          8
GPU Count:          0
Memory Avail:       4.43 GB / 15.70 GB (28.2%)
Disk Space Avail:   36.31 GB / 475.50 GB (7.6%)
Setting presets to: bolt_base

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'D',
 'hyperparameters': {'Chronos': {'fine_tune': True,
                                 'fine_tune_lr': 1e-05,
                                 'fine_tune_steps': 7000}},
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 10,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': True

In [8]:
predictions = predictor.predict(train_data)
predictor.plot(
    data=data,
    predictions=predictions,
    item_ids=data.item_ids[:2],
    max_history_length=200,
);

Model not specified in predict, will default to the model with the best validation score: Chronos[autogluon__chronos-bolt-small]


In [9]:
%matplotlib inline
predictions = predictor.predict(train_data)
predictor.plot(
    data=data,
    predictions=predictions,
    item_ids=data.item_ids[:2],
    max_history_length=200,
);

Model not specified in predict, will default to the model with the best validation score: Chronos[autogluon__chronos-bolt-small]


In [10]:
model_path = "./AutogluonModels/ag-20250623_071832/"

predictor = TimeSeriesPredictor.load(model_path)

Loading predictor from path c:\workspace\CrimeMap\AutogluonModels\ag-20250623_071832


In [11]:
predictions = predictor.predict(train_data)
predictor.plot(
    data=data,
    predictions=predictions,
    item_ids=data.item_ids[:2],
    max_history_length=200,
);

Model not specified in predict, will default to the model with the best validation score: Chronos[autogluon__chronos-bolt-small]
